# Quoridor GPU model comparison
Select **Runtime → Change runtime type → GPU**. Upload this notebook to Colab.

On your computer, run `python3 training/prepare_colab.py`, then upload the resulting `experiments/training/colab-source.zip` below. This includes your current uncommitted Python/C++ source; it excludes checkpoints, credentials, and the web app. No GitHub push or Node installation is required.

GPU type and runtime lifetime vary. Run the sweep first, then choose an affordable model and batches. Synthetic throughput does not measure playing strength.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, io, os, subprocess, sys
uploaded = files.upload()
source = uploaded.get('colab-source.zip')
if source is None:
    raise ValueError('Upload colab-source.zip generated by training/prepare_colab.py')
root = Path('/content/Quoridoraner')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(source)) as archive:
    for entry in archive.infolist():
        target = (root / entry.filename).resolve()
        if not target.is_relative_to(root.resolve()):
            raise ValueError('Unsafe archive path')
    archive.extractall(root)
os.chdir(root)
# Keep Colab's CUDA-enabled PyTorch; do not install the Mac requirements pin.
import torch, numpy
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print('PyTorch', torch.__version__, 'CUDA', torch.version.cuda)
print(torch.cuda.get_device_properties(0))
subprocess.run([sys.executable, 'training/native/build_direct.py'], check=True)


## Persistent outputs
Mount Drive before training so checkpoints survive runtime deletion. Each run contains two models, AdamW states, replay, and search trees; larger runs can produce large checkpoints. Saves to Drive count toward runtime, so keep the same checkpoint interval across comparisons. Model benchmarks use local storage.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_ROOT = Path('/content/drive/MyDrive/quoridor-experiments')
RUN_ROOT.mkdir(parents=True, exist_ok=True)


## Find a useful model and batch size
Each case runs in a separate process. OOM cases are recorded and the sweep continues. Timings include input/output transfers for inference and replay sampling/transfers for updates. Memory is per model; the trainer keeps **two** models and optimizers resident, plus CPU replay/search trees. Leave substantial headroom. BF16 cases are skipped as unsupported on GPUs without BF16 support.


In [ ]:
subprocess.run([sys.executable, 'training/benchmark_cuda.py',
    '--out', str(RUN_ROOT / 'cuda-sweep.json'),
    '--architectures', 'small', 'residual64', 'residual128', 'residual256', 'residual512',
    '--batches', '32', '128', '512', '--precisions', 'fp32', 'bf16'], check=True)


In [ ]:
import json, pandas as pd
report = json.loads((RUN_ROOT / 'cuda-sweep.json').read_text())
print(report['hardware'])
pd.json_normalize(report['rows'])[['architecture', 'precision', 'batch', 'status',
    'parameters', 'inference.positions_per_second', 'update.positions_per_second',
    'update.peak_allocated_bytes']]


## Compare learning at equal work
Edit the model list after inspecting the sweep. Start with one seed and a short run, then use several paired seeds and longer budgets. These defaults are a pilot, not a claim of convergence. All architectures get the same games, search simulations, minibatch size, updates, openings, curriculum, and evaluation. Each run trains both the selfplay and negamax arms. Reusing a run name resumes it; changing architecture or precision requires a new name.

The CPU performs tree search, legality checks, and negamax. Increase `parallel_games` together with `games` to increase inference batches, then inspect `performance.json`; Colab CPU or RAM limits can become the bottleneck. Increasing only optimizer batch size does not increase MCTS inference batches. Keep batch settings identical for the controlled comparison.


In [ ]:
architectures = ['small', 'residual64', 'residual128']  # add residual256/512 after checking memory
seeds = [1]  # use [1, 2, 3] for stronger evidence
iterations = 20
parallel_games = 128
batch_size = 256
workers = min(4, os.cpu_count() or 1)
precision = 'bf16' if torch.cuda.is_bf16_supported() else 'fp32'
for seed in seeds:
    order = architectures if seed % 2 else list(reversed(architectures))
    for architecture in order:
        directory = RUN_ROOT / f'{architecture}-{precision}-seed{seed}'
        if (directory / 'latest.pt').exists():
            command = [sys.executable, 'training/train.py', '--resume', str(directory),
                       '--iterations', str(iterations), '--device', 'cuda', '--plain']
        else:
            command = [sys.executable, 'training/train.py', '--out', str(directory),
                '--architecture', architecture, '--precision', precision,
                '--device', 'cuda', '--engine', 'direct', '--workers', str(workers),
                '--iterations', str(iterations), '--seed', str(seed),
                '--games', str(parallel_games), '--parallel-games', str(parallel_games),
                '--batch-size', str(batch_size), '--updates', '32', '--simulations', '64',
                '--curriculum', 'races', '--curriculum-interval', '25',
                '--eval-games', '20', '--eval-every', '10', '--heldout-games', '40',
                '--checkpoint-seconds', '120', '--plain']
        print(' '.join(command), flush=True)
        subprocess.run(command, check=True)


## Review results
`summary.json` reports score, goal wins and artificial cutoffs separately; compare each arm against its matching baseline. `losses.json` reports fitting and `performance.json` reports throughput. Compare equal-work strength and elapsed time first. For equal-time comparisons, use separate fresh directories and the same external time budget per architecture; record actual games and updates because slower models complete less training. Early race curricula can yield high training scores without full-game strength.

For a frozen-target comparison independent of self-play data changes, upload a compatible `latest.pt` and use `training/benchmark_depth.py --phase frozen --source PATH --device cuda --architectures small residual64 residual128 --out PATH`. See `training/DEPTH_COMPARISON.md` for source/split requirements. That existing benchmark uses FP32. Keep precision identical within each comparison.

Ctrl-C/SIGINT in a terminal triggers a resumable save. A runtime being deleted may interrupt a save; Drive retains the last completed checkpoint. Colab's stop button is not a guarantee of graceful shutdown. Resume by rerunning the training cell after setup and Drive mounting.
